# Notebook 2 of 6 — II. Data cleaning and preparation
**Group XX** · Every decision here comes from an observation in notebook 01. Optional decisions are tested on the five validation windows before being kept.

| Step | From Part 1 (`notebooks/02_Preprocessing.ipynb`) | In Part 2 |
|---|---|---|
| Invalid values → missing | `clean_values` (week 3): negatives and the 1e9 placeholder become `NaN` | **Kept**, same rule |
| Empty rows | Kept in the data, although exploration said to remove them | **Changed:** dropped (no date, no information) |
| Missing values | Median imputation + a `_missing` flag for every numeric column | **Changed:** values that follow from a definition are rebuilt first; only `first_purchase` is kept as a flag, because Part 1's own `VarianceThreshold` (feature selection, 3.5.1) showed the other flags were constant inside the splits |
| New columns | `first_purchase`, `history_unknown`, `net_value_per_detail` | **Kept** `first_purchase` and value per line (`value_per_detail`); `history_unknown` only marked the empty rows, which are now dropped. **New:** 6 ratios and `purchase_month` |
| `log1p`, then median, then scaling; `__MISSING__` + one-hot for categories | `fill_missing` / `fold_matrices` from week 4 | **Kept**, same order |
| Fitting per split | Hand-written `fold_matrices(train, test)` | **Changed:** the same steps inside a scikit-learn `Pipeline` + `ColumnTransformer` (week 5), so they are refitted automatically in every split and the fitted object can predict `test.csv` |
| `clientes.csv`, `customer_id`, `random_noise`, `purchase_date` | Left out without a test | **Kept** out, but the `clientes.csv` join and the other optional choices are now **tested** (Section 2.14) |

In [ ]:
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression

import sys
import os

# Get the parent project directory
project_dir = os.path.abspath("..")

# Add it to Python's import path
sys.path.insert(0, project_dir)

# Now import your module
import helpers.project_pipeline as pp

import importlib
importlib.reload(pp)


# A category that a training split never saw is encoded as zeros; scikit-learn warns
# about it every time. That behaviour is intended (Section 2.13), so the warning is hidden.
warnings.filterwarnings("ignore", message="Found unknown categories")
pd.set_option("display.width", 160)

train, test, clientes = pp.load_raw(pp.find_data_folder())
# The validation design of notebook 01: 5 yearly windows, each trained on purchases more than
# 90 days earlier (Part 1 used the same 90-day gap), plus the last 12 months as a sealed holdout.
design = pp.development_split(train)
usable = train[~pp.empty_rows(train)]
print(f"Usable labelled rows: {len(usable)} (65 empty rows dropped) | development {len(design.X_dev)} | holdout {len(design.X_hold)}")

## 2.11 Stateless cleaning


> **Kept from Part 1:** the rule of `clean_values` (Part 1, Section 2; week 3): negative amounts and day counts and the 1e9 placeholder become missing.
>
> **Changed:** the 65 empty rows are dropped instead of kept.
>
> **New in Part 2:** missing values that follow from a definition are rebuilt instead of guessed (Part 1 filled them all with the median).

`clean_purchases` (in `project_pipeline.py`) applies the same rules to every row of train, test or a future purchase. It learns nothing from the data, so it cannot leak information:

1. the 1e9 placeholder and negative amounts or day counts become missing;
2. missing values that follow from a definition are rebuilt (average purchase value, average gap, days since the previous purchase for a single earlier purchase, `spend_365d` when the 365-day window is empty or holds the whole history).

The table counts what each rule changed.

In [ ]:
# Counts what each rule of pp.clean_purchases changed (the rules themselves are in the module).
def cleaning_report(frame):
    raw = frame[pp.RAW_NUMERIC]
    cleaned = pp.clean_purchases(frame)[pp.RAW_NUMERIC]
    invalid = (raw < 0) | (raw >= pp.SENTINEL)
    marked_or_missing = raw.isna() | invalid
    report = pd.DataFrame({
        "invalid -> missing": invalid.sum(),
        "missing before repair": marked_or_missing.sum(),
        "rebuilt from definition": (marked_or_missing & cleaned.notna()).sum(),
        "still missing": cleaned.isna().sum(),
    })
    return report[report.drop(columns="still missing").sum(axis=1) > 0]

print("Training rows:")
display(cleaning_report(usable))
print("Test rows:")
display(cleaning_report(test))

- `prior_mean_purchase_value`: all 32 invalid values and all 194 randomly missing ones get their true value back; the 603 values still missing are exactly the first purchases (structural).
- `prior_mean_gap_days`: 203 values are rebuilt; what remains is almost entirely customers with fewer than two earlier purchases.
- `spend_365d`: 82 of 225 values are rebuilt; `days_since_previous_purchase`: 22 single-history customers recover their value.
- What is still missing is structural (short histories) or random and is handled by the learned imputation in Section 2.13. The same rules change the test rows in the same way, without looking at any training statistic.

## 2.12 Engineered features


> **Kept from Part 1:** `first_purchase` and value per order line (Part 1: `first_purchase`, `net_value_per_detail`).
>
> **Changed:** Part 1's 28 `_missing` flags are not used: its own `VarianceThreshold` test showed they were constant inside the time splits.
>
> **New in Part 2:** the rhythm ratios and `purchase_month`, each motivated by a finding of notebook 01.

Each new feature is a ratio of columns in the same row (so it only uses the past) and expresses an idea from the exploration more directly than the raw columns:

| Feature | Definition | Idea |
|---|---|---|
| `first_purchase` | `prior_purchase_count == 0` | A first purchase has no history (structural missingness flag) |
| `recency_ratio` | days since previous / average gap | Above 1: the customer is already late for their usual rhythm |
| `purchases_per_year` | prior purchases / tenure in years | Long-run buying frequency |
| `ticket_vs_history` | current net value / average past value | Unusually large orders may cover a longer period |
| `recent_activity_share` | purchases in last 90 days / last 365 days | Is the customer speeding up or slowing down? |
| `discount_rate` | discounts / gross value | Commercial conditions of this sale |
| `tax_rate` | tax / net value | 0 for export customers (notebook 01, Section 2.9) |
| `value_per_detail` | net value / number of lines | Size of each order line |
| `purchase_month` | calendar month (category) | Four-month buying cycle (notebook 01, Section 2.6) |

Skewed amounts, counts and day columns, and the skewed ratios, receive `log1p`, which keeps every row but compresses the long tails seen in notebook 01.

In [ ]:
# Week 4: relevance = |point-biserial r| with the target (same index Part 1 used in 3.5.3).
# Relevance of each engineered feature on the development rows, next to the strongest raw columns
# (|point-biserial r| after cleaning and log1p; descriptive only, not used to choose anything).
prepared = pp.prepare_frame(design.X_dev)
numeric = [c for c in prepared.select_dtypes("number").columns]
relevance = pd.Series(pp.target_correlation(prepared[numeric].fillna(prepared[numeric].median()), design.y_dev),
                      index=numeric).sort_values(ascending=False)
view = relevance.to_frame("|r| with target")
view["engineered"] = view.index.isin(list(pp.ENGINEERED))
display(view.head(12).round(3))
display(view[view["engineered"]].round(3))

`purchases_per_year` ranks fourth of all columns (|r| = 0.37), next to the rolling purchase counts. `discount_rate` (0.22) is clearly more informative than any raw current-purchase amount, probably because discounts mark established accounts. `first_purchase` and `recent_activity_share` are moderately related; `recency_ratio` and `ticket_vs_history` are weak on their own. A linear, one-column score cannot see thresholds or interactions (for example "late *and* usually frequent"), so it is only a first look: whether the new features help a model is tested in Section 2.14 and decided by feature selection in notebook 03.

## 2.13 Learned preprocessing (fitted only on training rows)


> **Kept from Part 1:** `log1p` before median imputation, scaling of numeric columns, an explicit missing label and one-hot encoding for categories (Part 1's `fill_missing` and `fold_matrices`, week 4).
>
> **Changed:** the steps are a scikit-learn `Pipeline` with a `ColumnTransformer` (week 5) instead of a hand-written function, so each split refits them and the same fitted object later predicts `test.csv`. The encoder uses `handle_unknown="infrequent_if_exist"`: with the default `min_frequency=1` it behaves exactly like Part 1's `"ignore"` (unseen level → zeros).

`build_preprocessor()` chains the stateless steps with the steps that learn from data:

| Block | Step | Learned from |
|---|---|---|
| all | `clean_purchases`, `add_features`, `log1p`, drop identifiers, noise and exact-sum columns | nothing (stateless) |
| numeric | median imputation, then `StandardScaler` | training rows of the split |
| categorical | `Missing` level, then one-hot encoding (`min_frequency` merges rare levels) | training rows of the split |

Scaling is needed by the distance- and penalty-based models (KNN, logistic regression, MLP) and does not change tree models, so one recipe serves every algorithm. Because the whole recipe sits inside a scikit-learn `Pipeline`, `fit` learns medians, levels and scales from the training rows only, and `transform` applies them unchanged to validation, holdout or test rows.

In [ ]:
# fit() learns medians, scales and category levels from these rows only (week 4: fit on train,
# transform the rest); transform() on test reuses them unchanged.
preprocessor = pp.build_preprocessor().fit(design.X_dev, design.y_dev)
A_dev = preprocessor.transform(design.X_dev)
A_test = preprocessor.transform(test)                     # same fitted values, nothing refitted
names = preprocessor.named_steps["columns"].get_feature_names_out()
print(f"Development rows -> {A_dev.shape}, test rows -> {A_test.shape}")
print(f"Missing or infinite values left: development {(~np.isfinite(A_dev)).sum()}, test {(~np.isfinite(A_test)).sum()}")
blocks = pd.Series([("categorical" if any(n.startswith(c + "_") for c in pp.CATEGORICAL + [pp.MONTH]) else "numeric")
                    for n in names]).value_counts()
display(blocks.rename("prepared columns").to_frame())

# Each fold refits the recipe on its own training rows, so even the number of columns is learned.
widths = [pp.build_preprocessor().fit(design.X_dev.iloc[a], design.y_dev.iloc[a]).transform(design.X_dev.iloc[:1]).shape[1]
          for a, _ in design.folds]
print("Prepared columns per fold:", widths)

The development rows become a complete numeric matrix and the test rows receive exactly the same columns. Most columns are one-hot levels (mainly `transport_method`). The width grows from fold to fold because later training periods contain more transport and payment levels, which confirms that the encoder learns its categories inside each training split.

## 2.14 Do the optional decisions pay off?


> **Changed:** Part 1 decided to leave out `clientes.csv` and to add its flags without measuring the effect. Here every optional choice is measured on the validation windows (week 4: a change has to prove itself on held-out data) with `paired_delta` and the 2 × SEM rule, the same comparison Part 1 used in feature selection.

Each optional choice is compared with the full recipe on the five validation windows, with two reference models from different families (week 4: every change has to prove itself on held-out data). The score is ROC-AUC; a difference counts only if it is larger than twice its standard error across the windows.

In [ ]:
# Customer table candidates: country, commercial region and years as a customer at the time of purchase.
client_columns = clientes[[pp.CUSTOMER, "country_or_region", "commercial_region", "customer_since_year"]]

def with_clientes(X):
    joined = X.merge(client_columns, on=pp.CUSTOMER, how="left")
    joined["years_as_customer"] = joined[pp.DATE].dt.year - joined["customer_since_year"]
    joined.index = X.index
    return joined

X_dev_joined = with_clientes(design.X_dev)

# Two reference models from different families (linear and tree ensemble), because a choice can
# help one family and not the other; the final algorithm is only chosen in notebook 04.
references = {
    "LogisticRegression": LogisticRegression(max_iter=5000, random_state=pp.RANDOM_STATE),
    "ExtraTrees": ExtraTreesClassifier(n_estimators=300, min_samples_leaf=2, random_state=pp.RANDOM_STATE, n_jobs=1),
}
variants = {   # name: (rows, preprocessing options)
    "full recipe (reference)": (design.X_dev, {}),
    "+ clientes.csv columns": (X_dev_joined, {"extra_numeric": ["years_as_customer"],
                                              "extra_categorical": ["country_or_region", "commercial_region"]}),
    "- purchase_month": (design.X_dev, {"drop": (pp.MONTH,)}),
    "- engineered ratios": (design.X_dev, {"drop": tuple(pp.ENGINEERED)}),
    "rare levels merged (< 30 rows)": (design.X_dev, {"min_frequency": 30}),
}
rows = []
for model_name, model in references.items():
    reference_scores = None
    for variant, (X, options) in variants.items():
        fold_scores, _ = pp.cross_validate_time(pp.make_pipeline(model, **options), X, design.y_dev, design.folds)
        if reference_scores is None:
            reference_scores = fold_scores
        rows.append({"model": model_name, "variant": variant, "ROC-AUC": fold_scores["ROC_AUC"].mean(),
                     **pp.paired_delta(fold_scores["ROC_AUC"], reference_scores["ROC_AUC"])})
decisions = pd.DataFrame(rows).set_index(["model", "variant"])
display(decisions.round(4))

| Choice | Evidence | Decision |
|---|---|---|
| Join `clientes.csv` | −0.0013 (LogReg) and +0.0016 (ExtraTrees), both inside the noise | **Not joined.** No measurable gain, and it would add a dependency on a table whose update date is unknown plus customer-identifying fields |
| `purchase_month` | Removing it costs ExtraTrees 0.0053 AUC on all 5 windows; neutral for LogReg | **Kept** |
| Engineered ratios | Removing them changes nothing measurable (inside the noise for both) | **Kept as candidates**: feature selection (notebook 03) decides which survive, as engineering must come before selection |
| Merge levels seen in < 30 rows | Worse for both models (−0.0084 and −0.0014, 0/5 wins) | **No merging by default** (`min_frequency=1`); the setting stays in the tuning search of notebook 04 |

The rare transport levels help even on *later* windows: they carry information about specific customers who keep buying, and the chronological validation shows that this information still holds in the future. The cost of this choice is discussed in notebook 05.

## 2.15 The preprocessing recipe carried forward


> **Kept from Part 1:** the idea of Part 1's Section 9: the recipe is a strategy refitted inside every split, not a matrix prepared once.

| Step | Rule | Why |
|---|---|---|
| Drop 65 empty rows | Every purchase field and the date missing | No information, cannot be placed in time |
| Invalid values → missing | 1e9 placeholder, negative amounts or days | Impossible values (train and test) |
| Rebuild from definitions | average value, average gap, days since previous, `spend_365d` | Restores the true value instead of a guess |
| Drop exact sums | `Total a Pagar`, `Total Bruto` | Pure redundancy |
| Exclude non-features | `ID`, `customer_id`, `purchase_date`, `random_noise` | Identifiers, split variable, noise |
| Engineered features | 8 ratios/flags + `purchase_month` | Express rhythm, trend and seasonality directly |
| `log1p` | Skewed amounts, counts, days and ratios | Long right tails, keeps every row |
| Median + scaling (fitted per split) | Numeric columns | Simple, robust; needed by KNN, MLP, logistic regression |
| `Missing` level + one-hot (fitted per split) | Categorical columns, `min_frequency` tuned in notebook 04 | Nominal categories, unseen levels → zeros |

The recipe is not fitted here once and saved: it is a **strategy** that every later notebook refits inside each training split, exactly as the final model refits it on all training rows before predicting `test.csv`.